# Source Data Model in Full Refresh



Dit notebook voert de Full Refresh strategie uit:

- Stap 1: Alle SDM-tabellen leegmaken (DELETE)

- Stap 2: Data uit alle 5 operationele DB's opnieuw inlezen in het SDM met INSERT



Inlaadstrategie voor SCD:

- SDM: full refresh. Het SDM bewaart geen historie en voert zelf geen SCD Type 1 of Type 2 uit.

- DWH SCD Type 1: gewijzigde dimensierij met dezelfde Business Key wordt overschreven.

- DWH SCD Type 2: oude versie krijgt een eindtijd en een nieuwe versie krijgt een nieuwe Surrogate Key.

- Feiten bij SCD Type 2: een nieuwe feitrij gebruikt in het DWH de meest actuele Surrogate Key die hoort bij de Business Key uit het SDM.



Opdrachtcontrole:

- Ja, deze opdracht is al afgedekt in DWH.ipynb.

- Test 3 wijzigt eerst de klant met Business Key `klantnr = 1`.

- Test 5 voegt daarna een nieuwe feitrij toe in het SDM die weer verwijst naar `klantnr = 1`.

- Bij een SCD Type 2 DWH krijgt die nieuwe feitrij daarna de nieuwste actuele `klant_sk`.

- Die surrogate-key verwachting geldt alleen wanneer het actieve DWH op het SCD Type 2 schema draait.



**Let op**: draai dit notebook van boven naar beneden voor een volledige bron-refresh. Voor de SCD-validatie gebruik je daarna DWH.ipynb Test 3 en Test 5.


In [15]:
# 1. IMPORTS EN INSTELLINGEN
import sys
import sqlite3
import pandas as pd
from pathlib import Path
from IPython.display import display
from loguru import logger

log_path = Path("../logs/sdm_load.log")
log_path.parent.mkdir(parents=True, exist_ok=True)
LOG_FORMAT = "{time:YYYY-MM-DD HH:mm:ss}|{level}|{message}"

logger.remove()
logger.add(
    sys.stderr,
    level="INFO",
    format=LOG_FORMAT,
)
logger.add(
    log_path,
    level="INFO",
    format=LOG_FORMAT,
    encoding="utf-8",
    backtrace=True,
    diagnose=False,
)

# Database paden
DB_ACCESSOIREVERKOOP = "BikeToDrive_1_Accessoireverkoop.db"
DB_FIETSVERKOOP      = "BikeToDrive_2_Fietsverkoop.db"
DB_ONDERHOUD         = "BikeToDrive_3_Onderhoud.db"
DB_ACCESSOIRE_INKOOP = "BikeToDrive_4_Accessoire_Inkoop.db"
DB_FIETS_INKOOP      = "BikeToDrive_5_Fiets_Inkoop.db"
DB_SDM               = "BikeToDrive_6 _SourceDataModel.db"

logger.info(f"SDM_REFRESH|Initialize databases: {DB_SDM}")

2026-04-22 00:38:21|INFO|SDM_REFRESH|Initialize databases: BikeToDrive_6 _SourceDataModel.db


In [16]:
# 2. TABLE MAPPING CONFIGURATIE
# Elke tuple: (bron_db, bron_tabel, sdm_tabel)
# Volgorde = insert-volgorde (ouder-tabellen eerst voor FK-veiligheid)

TABLE_MAPPING = [
    # Accessoireverkoop
    (DB_ACCESSOIREVERKOOP, "Leverancier",       "Accessoire_Verkoop_Leverancier"),
    (DB_ACCESSOIREVERKOOP, "Accessoire",         "Accessoire_Verkoop_Accessoire"),
    (DB_ACCESSOIREVERKOOP, "Filiaal",            "Accessoire_Verkoop_Filiaal"),
    (DB_ACCESSOIREVERKOOP, "Monteur",            "Accessoire_Verkoop_Monteur"),
    (DB_ACCESSOIREVERKOOP, "Klant",              "Accessoire_Verkoop_Klant"),
    (DB_ACCESSOIREVERKOOP, "Accessoire_Verkoop", "Accessoire_Verkoop"),

    # Fietsverkoop
    (DB_FIETSVERKOOP, "Fabrikant",    "Fiets_Verkoop_Fabrikant"),
    (DB_FIETSVERKOOP, "Fiets",        "Fiets_Verkoop_Fiets"),
    (DB_FIETSVERKOOP, "Filiaal",      "Fiets_Verkoop_Filiaal"),
    (DB_FIETSVERKOOP, "Monteur",      "Fiets_Verkoop_Monteur"),
    (DB_FIETSVERKOOP, "Klant",        "Fiets_Verkoop_Klant"),
    (DB_FIETSVERKOOP, "Fiets_Verkoop","Fiets_Verkoop"),

    # Onderhoud
    (DB_ONDERHOUD, "Fabrikant", "Onderhoud_Fabrikant"),
    (DB_ONDERHOUD, "Fiets",     "Onderhoud_Fiets"),
    (DB_ONDERHOUD, "Filiaal",   "Onderhoud_Filiaal"),
    (DB_ONDERHOUD, "Monteur",   "Onderhoud_Monteur"),
    (DB_ONDERHOUD, "Onderhoud", "Onderhoud"),

    # Accessoire Inkoop
    (DB_ACCESSOIRE_INKOOP, "Leverancier",      "Accessoire_Inkoop_Leverancier"),
    (DB_ACCESSOIRE_INKOOP, "Accessoire",        "Accessoire_Inkoop_Accessoire"),
    (DB_ACCESSOIRE_INKOOP, "Accessoire_Inkoop", "Accessoire_Inkoop"),

    # Fiets Inkoop
    (DB_FIETS_INKOOP, "Fabrikant",    "Fiets_Inkoop_Fabrikant"),
    (DB_FIETS_INKOOP, "Fiets",        "Fiets_Inkoop_Fiets"),
    (DB_FIETS_INKOOP, "Fiets_Inkoop", "Fiets_Inkoop"),
]

logger.info(f"SDM_REFRESH|Table map ready: {len(TABLE_MAPPING)} tables")

2026-04-22 00:38:24|INFO|SDM_REFRESH|Table map ready: 23 tables


In [17]:
# 3. GENERIEKE FUNCTIES

def reset_sdm(conn):
    """Verwijdert alle data uit SDM tabellen (omgekeerde insert-volgorde voor FK-veiligheid)."""
    logger.info("SDM_REFRESH|Start reset")
    for _, _, sdm_tabel in reversed(TABLE_MAPPING):
        sql_delete = f"DELETE FROM {sdm_tabel}"
        try:
            logger.info(f"SDM_REFRESH|SQL: {sql_delete}")
            conn.execute(sql_delete)
        except sqlite3.Error as e:
            logger.error(f"SDM_REFRESH|SQL failed: {sql_delete} ({e})")
    conn.commit()
    logger.success("SDM_REFRESH|Reset done")


def load_db_to_sdm(sdm_conn):
    """Laadt alle bron-tabellen in het SDM via TABLE_MAPPING (pandas to_sql)."""
    logger.info("SDM_REFRESH|Start load")
    huidig_db = None
    for src_db, src_tabel, sdm_tabel in TABLE_MAPPING:
        if src_db != huidig_db:
            logger.info(f"SDM_REFRESH|Source database: {src_db}")
            huidig_db = src_db
        try:
            sql_select = f"SELECT * FROM {src_tabel}"
            with sqlite3.connect(src_db) as src_conn:
                logger.info(f"SDM_REFRESH|SQL: {sql_select}")
                df = pd.read_sql_query(sql_select, src_conn)

            logger.info(f"SDM_REFRESH|SQL: INSERT INTO {sdm_tabel} (...) VALUES (...) [{len(df)} rows]")
            df.to_sql(sdm_tabel, sdm_conn, if_exists="append", index=False)
        except Exception as e:
            logger.error(f"SDM_REFRESH|SQL failed: INSERT INTO {sdm_tabel} ({e})")
    sdm_conn.commit()
    logger.success("SDM_REFRESH|Load done")


logger.success("SDM_REFRESH|Functions ready")

2026-04-22 00:38:28|SUCCESS|SDM_REFRESH|Functions ready


In [18]:
# 4. MAIN SCRIPT - Full Refresh uitvoeren

logger.info("SDM_REFRESH|Start full refresh")

try:
    with sqlite3.connect(DB_SDM) as sdm_conn:
        logger.info("SDM_REFRESH|Start step: reset")
        reset_sdm(sdm_conn)
        logger.success("SDM_REFRESH|Step done: reset")

        logger.info("SDM_REFRESH|Start step: load sources")
        load_db_to_sdm(sdm_conn)
        logger.success("SDM_REFRESH|Step done: load sources")

    logger.success("SDM_REFRESH|Full refresh done")

except Exception as e:
    logger.exception(f"SDM_REFRESH|Full refresh failed ({e})")

2026-04-22 00:38:31|INFO|SDM_REFRESH|Start full refresh
2026-04-22 00:38:31|INFO|SDM_REFRESH|Start step: reset
2026-04-22 00:38:31|INFO|SDM_REFRESH|Start reset
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Fiets_Inkoop
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Fiets_Inkoop_Fiets
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Fiets_Inkoop_Fabrikant
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Accessoire_Inkoop
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Accessoire_Inkoop_Accessoire
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Accessoire_Inkoop_Leverancier
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Onderhoud
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Onderhoud_Monteur
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Onderhoud_Filiaal
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Onderhoud_Fiets
2026-04-22 00:38:31|INFO|SDM_REFRESH|SQL: DELETE FROM Onderhoud_Fabrikant
2026-04-22 00:38:31|INFO|SDM_REF



##  Reset SDM functie uitvoeren


In [ ]:
# Reset SDM (alleen leegmaken, geen data inladen)

logger.info("SDM_REFRESH|Start reset only")

try:
    with sqlite3.connect(DB_SDM) as sdm_conn:
        reset_sdm(sdm_conn)
    logger.success("SDM_REFRESH|Reset only done")

except Exception as e:
    logger.exception(f"SDM_REFRESH|Reset only failed ({e})")

2026-04-15 12:58:40.583 | INFO     | __main__:<module>:3 - START RESET SDM
2026-04-15 12:58:40.585 | INFO     | __main__:reset_sdm:8 - ✓ reset: Fiets_Inkoop
2026-04-15 12:58:40.586 | INFO     | __main__:reset_sdm:8 - ✓ reset: Fiets_Inkoop_Fiets
2026-04-15 12:58:40.587 | INFO     | __main__:reset_sdm:8 - ✓ reset: Fiets_Inkoop_Fabrikant
2026-04-15 12:58:40.587 | INFO     | __main__:reset_sdm:8 - ✓ reset: Accessoire_Inkoop
2026-04-15 12:58:40.588 | INFO     | __main__:reset_sdm:8 - ✓ reset: Accessoire_Inkoop_Accessoire
2026-04-15 12:58:40.588 | INFO     | __main__:reset_sdm:8 - ✓ reset: Accessoire_Inkoop_Leverancier
2026-04-15 12:58:40.589 | INFO     | __main__:reset_sdm:8 - ✓ reset: Onderhoud
2026-04-15 12:58:40.589 | INFO     | __main__:reset_sdm:8 - ✓ reset: Onderhoud_Monteur
2026-04-15 12:58:40.590 | INFO     | __main__:reset_sdm:8 - ✓ reset: Onderhoud_Filiaal
2026-04-15 12:58:40.590 | INFO     | __main__:reset_sdm:8 - ✓ reset: Onderhoud_Fiets
2026-04-15 12:58:40.591 | INFO     | __ma



##  Check of data correct in SDM zit


In [13]:
# 5. VERIFICATIE - Aantal rijen per tabel als DataTable

logger.info("SDM_REFRESH|Start verification")

with sqlite3.connect(DB_SDM) as sdm_conn:
    rows = []
    for _, _, sdm_tabel in TABLE_MAPPING:
        try:
            count = pd.read_sql_query(
                f"SELECT COUNT(*) AS aantal FROM {sdm_tabel}", sdm_conn
            ).iloc[0, 0]
            rows.append({"Tabel": sdm_tabel, "Rijen": count, "Status": "✓" if count > 0 else "⚠ leeg"})
        except Exception as e:
            rows.append({"Tabel": sdm_tabel, "Rijen": "–", "Status": f"✗ {e}"})

df_check = pd.DataFrame(rows)
display(df_check.style.set_caption("SDM Verificatie — Rijen per tabel").hide(axis="index"))

logger.success(f"SDM_REFRESH|Verification done: {len(rows)} tables")

2026-04-22 00:34:42|INFO|SDM_REFRESH|Start verification


Tabel,Rijen,Status
Accessoire_Verkoop_Leverancier,5,✓
Accessoire_Verkoop_Accessoire,10,✓
Accessoire_Verkoop_Filiaal,4,✓
Accessoire_Verkoop_Monteur,10,✓
Accessoire_Verkoop_Klant,20,✓
Accessoire_Verkoop,100,✓
Fiets_Verkoop_Fabrikant,10,✓
Fiets_Verkoop_Fiets,75,✓
Fiets_Verkoop_Filiaal,4,✓
Fiets_Verkoop_Monteur,10,✓


2026-04-22 00:34:42|SUCCESS|SDM_REFRESH|Verification done: 23 tables


In [ ]:
# 6. PREVIEW - Bekijk de eerste rijen van een SDM tabel
# Wijzig PREVIEW_TABLE naar de tabel die je wilt inspecteren

PREVIEW_TABLE = "Fiets_Inkoop"

with sqlite3.connect(DB_SDM) as sdm_conn:
    df_preview = pd.read_sql_query(f"SELECT * FROM {PREVIEW_TABLE} LIMIT 10", sdm_conn)

logger.success(f"SDM_REFRESH|Preview ready: {PREVIEW_TABLE} ({len(df_preview)} rows)")
display(df_preview.style.set_caption(f"Preview: {PREVIEW_TABLE} (eerste 10 rijen)").hide(axis="index"))

2026-04-15 12:58:40.686 | INFO     | __main__:<module>:9 - Preview 'Fiets_Inkoop' — 0 rijen getoond


inkoopnr,inkoopmaand,inkoopjaar,aantal,fiets
